# [Evaluación a la asesoría del Profe Marco]()

# Implementación de la recomendación 2 



Para aplicar la **Recomendación 2**, usaremos un enfoque híbrido muy común en la econometría de alta dimensionalidad: **Selección de Características basada en Regularización (Lasso)** como un paso de filtrado previo (*feature screening*) antes de pasar las variables al modelo ARIMAX.



# ¿Por qué hacer esto?



La regresión **Lasso (Regularización $L_1$)** añade una penalización a la magnitud de los coeficientes de regresión. 

Cuando el parámetro de penalización ($\alpha$) es el adecuado, Lasso fuerza a que los coeficientes de las variables irrelevantes o altamente redundantes sean **exactamente cero**.



En tu caso, en lugar de usar PCA (que transforma las variables en componentes abstractos), Lasso mantendrá las variables originales (por ejemplo, sabrás exactamente que sobrevivió `temp_lag_3` o `prec_lag_2`) y descartará todo el ruido meteorológico restante, dejando una matriz exógena limpia para que el optimizador del ARIMAX converja velozmente y sin *overfitting*.



Aquí tienes el algoritmo de Python reconfigurado bajo esta estrategia:



In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_absolute_error

# 1. Configuración de rutas de archivos y directorios
ruta_entrada = r"C:\Users\marco\Documentos\investigacion\arima\06_entrenar_modelo\0_obtener_rezagos\2_datos\1_raw\meteo_epi_2021-2026_1.xlsx"
directorio_resultados = r"C:\Users\marco\Documentos\investigacion\rolling_forecast\3_resultados"
os.makedirs(directorio_resultados, exist_ok=True)

# 2. Cargar datos y ordenar cronológicamente
print("Cargando y preparando el dataset original...")
df = pd.read_excel(ruta_entrada)
df['fecha'] = pd.to_datetime(df['fecha'])
df = df.sort_values('fecha').reset_index(drop=True)

# 3. Generar todos los 12 rezagos originales (para buscar el filtrado Lasso)
print("Generando la matriz completa de rezagos...")
columnas_excluidas = ['fecha', 'semana_epi', 'año']
columnas_con_rezago = [col for col in df.columns if col not in columnas_excluidas]

nuevas_columnas = {}
for col in columnas_con_rezago:
    for lag in range(1, 13):
        nuevas_columnas[f"{col}_lag_{lag}"] = df[col].shift(lag)

df_rezagos = pd.DataFrame(nuevas_columnas, index=df.index)
df_procesado = pd.concat([df, df_rezagos], axis=1).dropna().reset_index(drop=True)

# Separar variables predictoras crudas y objetivo
columnas_candidatas = [col for col in df_procesado.columns if col not in ['fecha', 'casos_dengue']]
X_cruda = df_procesado[columnas_candidatas]
y = df_procesado['casos_dengue']
fechas = df_procesado['fecha']

# 4. Selección de Variables mediante Regresión Lasso (con Validación Cruzada)
print("Ejecutando LassoCV para identificar rezagos y variables clave...")
# Es mandatorio estandarizar los datos para que Lasso penalice de forma equitativa
scaler = StandardScaler()
X_cruda_escalada = scaler.fit_transform(X_cruda)

# LassoCV encuentra el alpha (penalización) óptimo automáticamente mediante validación cruzada
lasso = LassoCV(cv=5, random_state=42, max_iter=10000)
lasso.fit(X_cruda_escalada, y)

# Filtrar las columnas cuyos coeficientes NO fueron llevados a cero por Lasso
coeficientes = lasso.coef_
columnas_seleccionadas = [col for col, coef in zip(columnas_candidatas, coeficientes) if coef != 0]

print(f"-> Lasso redujo tus predictores de {X_cruda.shape[1]} a {len(columnas_seleccionadas)} variables clave.")
print(f"Variables supervivientes: {columnas_seleccionadas}")

# Filtrar nuestra matriz X con los sobrevivientes de Lasso
X_filtrada = X_cruda[columnas_seleccionadas]

# 5. Simulación Móvil con Estrategia Rolling Forecast (ARIMAX con variables filtradas)
tamanio_entrenamiento_inicial = int(len(df_procesado) * 0.80)
predicciones_test = []
mae_entrenamiento_lista = []
indices_test = range(tamanio_entrenamiento_inicial, len(df_procesado))

print(f"Iniciando Rolling Forecast con ARIMAX (Filtro Lasso) para {len(indices_test)} pasos...")

orden_arima = (1, 1, 0)

for i in indices_test:
    X_train_fold = X_filtrada.iloc[:i]
    y_train_fold = y.iloc[:i]
    X_test_fold = X_filtrada.iloc[i:i+1]
    
    # Inicializar y ajustar el modelo ARIMAX óptimo
    modelo = ARIMA(endog=y_train_fold, exog=X_train_fold, order=orden_arima)
    modelo_ajustado = modelo.fit(method_kwargs={'maxiter': 150})
    
    # Evaluar rendimiento en entrenamiento
    pred_train_fold = modelo_ajustado.fittedvalues
    mae_train_paso = mean_absolute_error(y_train_fold[1:], pred_train_fold[1:])
    mae_entrenamiento_lista.append(mae_train_paso)
    
    # Proyección del siguiente paso
    pred_test_paso = modelo_ajustado.forecast(steps=1, exog=X_test_fold)
    predicciones_test.append(pred_test_paso.values[0])

# 6. Consolidación de Resultados y Métricas Globales
y_test_real = y.iloc[tamanio_entrenamiento_inicial:].values
predicciones_test = np.array(predicciones_test)
fechas_test = fechas.iloc[tamanio_entrenamiento_inicial:].reset_index(drop=True)

mae_test_global = mean_absolute_error(y_test_real, predicciones_test)
mae_train_global = np.mean(mae_entrenamiento_lista)

print(f"\n--- RENDIMIENTO GLOBAL (ENFOQUE LASSO) ---")
print(f"MAE Promedio Entrenamiento: {mae_train_global:.4f}")
print(f"MAE Global Testeo (Rolling): {mae_test_global:.4f}")

# 7. Exportación de Resultados a Excel
ruta_excel = os.path.join(directorio_resultados, "desempeno_rolling_forecast.xlsx")
with pd.ExcelWriter(ruta_excel, engine='openpyxl') as writer:
    df_metricas = pd.DataFrame({
        'Métrica': ['MAE Entrenamiento (Promedio)', 'MAE Testeo (Rolling Forecast)'],
        'Valor': [mae_train_global, mae_test_global]
    })
    df_metricas.to_excel(writer, sheet_name='Metricas_Globales', index=False)
    
    df_series = pd.DataFrame({
        'Fecha': fechas_test,
        'Casos_Reales': y_test_real,
        'Casos_Predichos': predicciones_test
    })
    df_series.to_excel(writer, sheet_name='Predicciones_Testeo', index=False)

print(f"Resultados guardados en Excel: {ruta_excel}")

# 8. Construcción de Gráficas Requeridas (Líneas y Barras)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

fechas_train = fechas.iloc[:tamanio_entrenamiento_inicial]
y_train_ultimo = y.iloc[:tamanio_entrenamiento_inicial]
modelo_final = ARIMA(endog=y_train_ultimo, exog=X_filtrada.iloc[:tamanio_entrenamiento_inicial], order=orden_arima).fit(method_kwargs={'maxiter': 150})

# Gráfico de Líneas - Entrenamiento
ax1.plot(fechas_train, y_train_ultimo, label='Real (Entrenamiento)', color='darkblue', alpha=0.7)
ax1.plot(fechas_train, modelo_final.fittedvalues, label='Ajustado ARIMAX (Lasso)', color='orange', linestyle='--', alpha=0.8)
ax1.set_title(f'Desempeño en Entrenamiento\n(MAE Promedio: {mae_train_global:.2f})')
ax1.set_xlabel('Fecha')
ax1.set_ylabel('Casos de Dengue')
ax1.legend()
ax1.tick_params(axis='x', rotation=45)

# Gráfico de Líneas - Testeo
ax2.plot(fechas_test, y_test_real, label='Real (Testeo)', color='darkgreen', marker='o', markersize=3)
ax2.plot(fechas_test, predicciones_test, label='Predicción Rolling Forecast (Lasso)', color='red', linestyle='--', marker='x', markersize=3)
ax2.set_title(f'Desempeño en Testeo Móvil\n(MAE Global: {mae_test_global:.2f})')
ax2.set_xlabel('Fecha')
ax2.set_ylabel('Casos de Dengue')
ax2.legend()
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(directorio_resultados, "comparativa_lineas_desempeno.png"), dpi=300)
plt.close()

# Gráfico de Barras - Comparativa de Errores MAE
plt.figure(figsize=(6, 5))
categorias = ['MAE Entrenamiento', 'MAE Testeo']
valores_mae = [mae_train_global, mae_test_global]
colores = ['#4C72B0', '#C44E52']

barras = plt.bar(categorias, valores_mae, color=colores, width=0.5, edgecolor='black')
plt.ylabel('Error Absoluto Medio (MAE)')
plt.title('Comparativa de Errores Globales (MAE) con Filtro Lasso')
plt.grid(axis='y', linestyle='--', alpha=0.7)

for barra in barras:
    yval = barra.get_height()
    plt.text(barra.get_x() + barra.get_width()/2.0, yval + (max(valores_mae)*0.01), f'{yval:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(directorio_resultados, "comparativa_barras_mae.png"), dpi=300)
plt.close()

print("¡Algoritmo finalizado con éxito! El cuello de botella de variables se resolvió usando regularización L1.")


Cargando y preparando el dataset original...
Generando la matriz completa de rezagos...
Ejecutando LassoCV para identificar rezagos y variables clave...
-> Lasso redujo tus predictores de 170 a 13 variables clave.
Variables supervivientes: ['año', 'temp_lag_1', 'hum_esp_lag_6', 'prec_lag_4', 'dias_lluvia_lag_10', 'vel_vi_min_lag_1', 'vel_vi_min_lag_8', 'vel_vi_min_lag_10', 'soi_lag_8', 'casos_dengue_lag_1', 'casos_dengue_lag_2', 'casos_dengue_lag_4', 'casos_dengue_lag_8']
Iniciando Rolling Forecast con ARIMAX (Filtro Lasso) para 54 pasos...

--- RENDIMIENTO GLOBAL (ENFOQUE LASSO) ---
MAE Promedio Entrenamiento: 5.5811
MAE Global Testeo (Rolling): 7.4598
Resultados guardados en Excel: C:\Users\marco\Documentos\investigacion\rolling_forecast\3_resultados\desempeno_rolling_forecast.xlsx
¡Algoritmo finalizado con éxito! El cuello de botella de variables se resolvió usando regularización L1.



# 